## Artificial Neural Network (ANN)

This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:  

- Start Simple: Begin with a simple architecture and gradually increase complexity if needed.  
- Grid Search/Random Search: Use grid search or random search to try different architectures.  
- Cross-Validation: Use cross-validation to evaluate the performance of different architectures.  
- Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:  
   - The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.  
   - A common practice is to start with 1-2 hidden layers.  

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle




In [2]:
data = pd.read_csv('Churn_Modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder = LabelEncoder()
data['Gender'] = label_encoder.fit_transform(data['Gender'])

onehot_encoder = OneHotEncoder()
geo_encoded = onehot_encoder.fit_transform(data[['Geography']])
geo_encoded_df = pd.DataFrame(geo_encoded.toarray(), columns = onehot_encoder.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis = 1)

X = data.drop('Exited', axis=1)
y = data['Exited']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
# save encoders and scaler
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
with open('onehot_encoder.pkl', 'wb') as f:
    pickle.dump(onehot_encoder, f)
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [5]:
# Define a function to create the Keras model

def create_model(neurons=32, layers=1):
    model=Sequential() #create a sequential model
    model.add(Dense(neurons,activation='relu',input_shape= (X_train.shape[1],))) #add the first layer with input shape
    
    for _ in range(layers-1): 
        model.add(Dense(neurons,activation='relu'))  #add additional layers based on the 'layers' parameter
    
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy']) #compile the model with Adam optimizer and binary crossentropy loss
    return model
    
    


In [10]:
# create a KerasClassifier wrapper for the model

model = KerasClassifier(layers=1, neurons=32, build_fn=create_model, verbose=1)

In [11]:
# Define the grid search parameters

param_grid = {
    'model__neurons': [16, 32, 64],
    'model__layers': [1, 2],
    'batch_size': [10, 20],
}

In [13]:
# perform grid search

from sklearn.model_selection import GridSearchCV


grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(X_train, y_train)

# summarize results
print(f"Best: {grid_result.best_score_:.4f} using {grid_result.best_params_}")
mean_test_scores = grid_result.cv_results_['mean_test_score']
params = grid_result.cv_results_['params']

c:\Users\krish\OneDrive\Desktop\AI Engineer\Deep Learning\ANN\ANN_Churn_Classification\venv\Lib\site-packages\scikeras\wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)






400/400 [==============================] - 2s 1ms/step - loss: 217.1253 - accuracy: 0.6820
Best: 0.7899 using {'batch_size': 20, 'model__layers': 2, 'model__neurons': 64}


In [16]:
# Print best result summary
print(f"Best Accuracy : {grid_result.best_score_:.4f}")
print(f"Best Params   : {grid_result.best_params_}")

# Extract the best Keras model from the KerasClassifier wrapper
best_model = grid_result.best_estimator_.model_

# Save the best model
best_model.save('HyperParameterTuning_ANN_best_model.h5')
print("\n✅ Best model saved as 'best_model.h5'")

Best Accuracy : 0.7899
Best Params   : {'batch_size': 20, 'model__layers': 2, 'model__neurons': 64}

✅ Best model saved as 'best_model.h5'


c:\Users\krish\OneDrive\Desktop\AI Engineer\Deep Learning\ANN\ANN_Churn_Classification\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
